<a href="https://colab.research.google.com/github/bvsvntv/ai-ml-learning/blob/main/notebooks/lang-graph/stm_trimming.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [143]:
!pip install -U langchain-openai langgraph-checkpoint-postgres psycopg2

In [144]:
from langgraph.graph import StateGraph, START, END, MessagesState
from langchain_openai import ChatOpenAI
from google.colab import userdata
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages.utils import trim_messages, count_tokens_approximately

In [145]:
api_key = userdata.get("OPENAI_API_KEY")
model = ChatOpenAI(api_key=api_key, model="gpt-6-luna")

In [146]:
MAX_TOKENS = 80

In [147]:
def call_model(state: MessagesState):

  # Trim conversation history -> last N messages that fit within the token limit
  messages = trim_messages(
      state["messages"],
      strategy="last",
      token_counter=count_tokens_approximately,
      max_tokens=MAX_TOKENS,
      # start_on="human",
      # include_system=True
  )

  print("Current token count: ", count_tokens_approximately(messages=messages))

  print("Messages after trimming.")
  print("-"*80)
  for message in messages:
    print(type(message).__name__, ":", message.content)
  print("-"*80)

  response = model.invoke(messages)

  return {"messages": [response]}

In [148]:
graph = StateGraph(MessagesState)
graph.add_node("call_model", call_model)

graph.add_edge(START, "call_model")
graph.add_edge("call_model", END)

In [149]:
checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [150]:
config1 = {"configurable": {"thread_id": "thread-1"}}
result = workflow.invoke({"messages": [{"role": "user", "content": "Hi, My name is James Bond."}]}, config=config1)

print("recent response: ",result["messages"][-1].content)

Current token count:  11
Messages after trimming.
--------------------------------------------------------------------------------
HumanMessage : Hi, My name is James Bond.
--------------------------------------------------------------------------------
recent response:  Hi James, nice to meet you! How can I help?


In [151]:
config1 = {"configurable": {"thread_id": "thread-1"}}
result = workflow.invoke({"messages": [{"role": "user", "content": "Am i relevant in this AI agent era?"}]}, config=config1)

print("recent response: ",result["messages"][-1].content)

Current token count:  40
Messages after trimming.
--------------------------------------------------------------------------------
HumanMessage : Hi, My name is James Bond.
AIMessage : Hi James, nice to meet you! How can I help?
HumanMessage : Am i relevant in this AI agent era?
--------------------------------------------------------------------------------
recent response:  Yes, James. Your relevance isn’t measured by how much you can compete with AI at its strongest tasks. It comes from what you know, the judgment you bring, the trust you build, and the choices you make about how to use these tools.

AI will change many jobs, and adapting will matter. A good starting point is to learn one AI tool that helps with your work, then focus on the parts that still need human context, responsibility, creativity, or connection. You don’t have to outpace every new tool—you just have to keep finding ways to make your experience count.


In [152]:
result = workflow.invoke({"messages": [{"role": "user", "content": "I'm learning about LangChain and LangGraph."}]}, config=config1)
print("recent response: ",result["messages"][-1].content)

Current token count:  15
Messages after trimming.
--------------------------------------------------------------------------------
HumanMessage : I'm learning about LangChain and LangGraph.
--------------------------------------------------------------------------------
recent response:  Great! A useful way to think about them:

- **LangChain** provides building blocks for LLM applications—models, prompts, tools, document retrieval, and integrations.
- **LangGraph** helps you orchestrate those components into **stateful workflows**, especially when you need branching, loops, human approval, or persistent agent state.

They’re complementary: you can use LangChain components inside a LangGraph workflow.

I can help with a beginner-friendly overview, a small example, or a comparison of when to use each. What are you working on?


In [153]:
result = workflow.invoke({"messages": [{"role": "user", "content": "What are LLM memory? And explain short term memory thoroughly with example?"}]}, config=config1)
print("recent response: ",result["messages"][-1].content)

Current token count:  23
Messages after trimming.
--------------------------------------------------------------------------------
HumanMessage : What are LLM memory? And explain short term memory thoroughly with example?
--------------------------------------------------------------------------------
recent response:  ## What does “memory” mean in an LLM?

An LLM’s **memory** is the information it can use when generating a response. It is not quite like human memory. In practice, there are two main kinds:

1. **Short-term memory:** Information available in the current conversation or task, usually because it is included in the model’s input.
2. **Long-term memory:** Information saved outside the current conversation—such as in a profile, database, or memory feature—and provided to the model again later when relevant.

The model’s **training** is different: training shapes the model’s general knowledge and behavior, but it usually does not mean the model can recall a particular user’s 

In [154]:
result = workflow.invoke({"messages": [{"role": "user", "content": "What is my name?"}]}, config=config1)
print("recent response: ",result["messages"][-1].content)

Current token count:  8
Messages after trimming.
--------------------------------------------------------------------------------
HumanMessage : What is my name?
--------------------------------------------------------------------------------
recent response:  I don’t know—you haven’t told me your name.


In [155]:
graph_state = workflow.get_state(config=config1)
values = graph_state.values
for message in values.get("messages", []):
  print(type(message).__name__, ": ", message.content)

HumanMessage :  Hi, My name is James Bond.
AIMessage :  Hi James, nice to meet you! How can I help?
HumanMessage :  Am i relevant in this AI agent era?
AIMessage :  Yes, James. Your relevance isn’t measured by how much you can compete with AI at its strongest tasks. It comes from what you know, the judgment you bring, the trust you build, and the choices you make about how to use these tools.

AI will change many jobs, and adapting will matter. A good starting point is to learn one AI tool that helps with your work, then focus on the parts that still need human context, responsibility, creativity, or connection. You don’t have to outpace every new tool—you just have to keep finding ways to make your experience count.
HumanMessage :  I'm learning about LangChain and LangGraph.
AIMessage :  Great! A useful way to think about them:

- **LangChain** provides building blocks for LLM applications—models, prompts, tools, document retrieval, and integrations.
- **LangGraph** helps you orchestr